In [22]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
import operator
load_dotenv()
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    
)

In [23]:
class Evaluation_Schema(BaseModel):
    feedback:str =Field(description="Detailed feedback for the essay")
    score:float =Field(description="Final score for the essay less than equal to 10 and greater than 0")
structured_model = model.with_structured_output(Evaluation_Schema)


class UPSCState(TypedDict):
    essay_text:str
    clarity_of_thoughts:str
    depth_of_analysis:str
    language:str
    summarized_feedback:str
    evaluation:str
    final_score_avg:float 
    individual_scores:Annotated[list[float],operator.add]

In [27]:
def Evaluate_COT(state:UPSCState)->UPSCState:
    essay = state["essay_text"]
    prompt=f"Evaluate the essay and give me the final feedback and score out of 10 in number , here is the essay:-  {essay}"
    clarity_of_thoughts = structured_model.invoke(prompt)
    return {"clarity_of_thoughts":clarity_of_thoughts.feedback,"individual_scores":[clarity_of_thoughts.score]}

def Evaluate_DOA(state:UPSCState)->UPSCState:
    essay = state["essay_text"]
    prompt=f"Evaluate the essay and give me the final feedback and score out of 10 in number , here is the essay:-  {essay}"
    depth_of_analysis = structured_model.invoke(prompt)
    return {"depth_of_analysis":depth_of_analysis.feedback,"individual_scores":[depth_of_analysis.score]}

def Evaluate_language(state:UPSCState)->UPSCState:
    essay = state["essay_text"]
    prompt=f"Evaluate the essay and give me the final feedback and score out of 10 in number , here is the essay:-  {essay}"
    language = structured_model.invoke(prompt)
    return {"language":language.feedback,"individual_scores":[language.score]}

def Evaluation_Score(state:UPSCState)->UPSCState:
    final_avg_Score = sum(state["individual_scores"])/len(state["individual_scores"])
    feedback_overall = state["depth_of_analysis"]+state["clarity_of_thoughts"]+state["language"]
    prompt = f"Give me the final feedback here is the overall feedback in 100 words only:- {feedback_overall}"
    final_feedback = model.invoke(prompt).content

    return {"summarized_feedback":final_feedback,"final_score_avg":final_avg_Score}
    

In [25]:
graph=StateGraph(UPSCState)

graph.add_node("Evaluate_clarity_of_thoughts",Evaluate_COT)
graph.add_node("Evaluate_depth_of_analysis",Evaluate_DOA)
graph.add_node("Evaluate_language",Evaluate_language)
graph.add_node("Final_Evaluation",Evaluation_Score)

#edges
graph.add_edge(START,"Evaluate_clarity_of_thoughts")
graph.add_edge(START,"Evaluate_depth_of_analysis")
graph.add_edge(START,"Evaluate_language")
graph.add_edge("Evaluate_clarity_of_thoughts","Final_Evaluation")
graph.add_edge("Evaluate_depth_of_analysis","Final_Evaluation")
graph.add_edge("Evaluate_language","Final_Evaluation")
graph.add_edge("Final_Evaluation",END)

workflow = graph.compile()

In [28]:
initial_state ={
    "essay_text":"""
The Case for Boredom

There is a particular kind of silence that modern life has almost engineered out of existence: the silence of having nothing to do. Waiting rooms now have televisions. Elevators play music. The moment a queue forms, a phone appears in every hand. We have become remarkably efficient at ensuring that boredom, once a routine feature of daily existence, rarely gets the chance to fully arrive.

This might seem like unambiguous progress. Boredom is uncomfortable, after all — a low hum of restlessness that most people would rather not sit with. But there is growing reason to think we have been too hasty in eliminating it, and that in doing so we have quietly given up something valuable.

Psychologists who study boredom describe it not as an absence of stimulation but as a specific kind of discomfort: the sense that our attention has nowhere satisfying to go. It is an unpleasant state, but unpleasant is not the same as useless. Discomfort is often the signal that prompts change. Hunger drives us to eat; loneliness drives us to seek company. Boredom, in this light, is the mind's way of saying that its current situation isn't using its capacities well — and that signal has historically pushed people toward something more meaningful, whether that's a new idea, a creative project, or simply a wandering thought that leads somewhere unexpected.

The trouble with eliminating boredom so thoroughly is that we may also be eliminating the conditions under which certain kinds of thinking happen. Research on mind-wandering suggests that when the brain isn't pinned down by an external task, it tends to drift toward autobiographical planning, problem-solving, and the loose association of ideas that often precedes a creative insight. Many people can point to a moment when a solution to a stubborn problem arrived not while they were working on it, but while they were doing something mundane — showering, walking, staring out a train window. These are precisely the moments that a constant stream of notifications and short-form content has crowded out.

There is also something to be said for boredom's role in shaping desire and attention. A person who is never bored never has to develop the internal resources to entertain themselves, to sit with an unstructured hour and decide, from scratch, what to do with it. Children in particular seem to need this. Overscheduled and constantly entertained, they can lose the muscle of self-directed play — the very thing that builds imagination and independence. An adult who has never learned to tolerate boredom may find themselves reaching for a screen not because they want to, but because they no longer know how to be alone with their own mind.

None of this is an argument for suffering, or for treating stimulation as inherently corrupting. Distraction has real value, and not every idle moment needs to be treated as a missed opportunity for insight. The point is narrower: that a life engineered to avoid boredom at every turn also avoids the conditions that produce reflection, patience, and originality. If we want more of those things, we may need to tolerate — even protect — a little more of the discomfort we've spent so much effort trying to erase.

The next time a spare five minutes appears and the impulse is to reach for a phone, it might be worth resisting, if only briefly. Not because boredom feels good, but because of what it sometimes makes possible."""
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'essay_text': "\nThe Case for Boredom\n\nThere is a particular kind of silence that modern life has almost engineered out of existence: the silence of having nothing to do. Waiting rooms now have televisions. Elevators play music. The moment a queue forms, a phone appears in every hand. We have become remarkably efficient at ensuring that boredom, once a routine feature of daily existence, rarely gets the chance to fully arrive.\n\nThis might seem like unambiguous progress. Boredom is uncomfortable, after all — a low hum of restlessness that most people would rather not sit with. But there is growing reason to think we have been too hasty in eliminating it, and that in doing so we have quietly given up something valuable.\n\nPsychologists who study boredom describe it not as an absence of stimulation but as a specific kind of discomfort: the sense that our attention has nowhere satisfying to go. It is an unpleasant state, but unpleasant is not the same as useless. Discomfort is often 